# **Hospital Readmission Risk Modeling**
## Final Report

### Author:  Bhavana Sanghi

### Project Goal: Predict 30-day hospital readmission risk to support targeted intervention and care management.

### This notebook summarizes the final methodology, results, interpretation, and model selection decisions for the hospital readmission risk modeling project.

## Executive Summary

The objective of this project was to build a predictive model to identify patients at elevated risk of 30-day hospital readmission so that healthcare providers can target post-discharge interventions more effectively.

The modeling pipeline began with logistic regression baselines, followed by L1-regularized logistic regression (LASSO) for feature selection and interpretability, and then a tuned XGBoost model to capture nonlinear relationships. Model performance was evaluated using ROC-AUC, recall, precision, F1-score, lift/gains analysis, calibration, SHAP-based interpretability, and fairness metrics across demographic groups.

The final comparison showed that linear models achieved similar discrimination performance (test ROC-AUC around 0.662), while tuned XGBoost improved ranking performance modestly (test ROC-AUC 0.6675) and substantially increased recall. A reduced-feature XGBoost model, created by removing low-importance feature groups identified through SHAP analysis, retained essentially identical performance (test ROC-AUC 0.6673) with fewer features.

The final selected model is the **reduced XGBoost model**, because it preserves predictive performance while improving simplicity and deployment practicality. Lift analysis showed that the model effectively concentrates readmissions in the highest-risk deciles, and fairness analysis showed no severe bias, though moderate disparities in false negative rates should be monitored in deployment.


## Problem Framing

Hospital readmissions are costly for both patients and healthcare systems. An effective readmission risk model can support targeted intervention by identifying patients who may benefit from additional follow-up, discharge planning, or care coordination.

This project is framed as a **risk prioritization problem**, not only a binary classification problem. In practice, the goal is not simply to assign a yes/no label, but to rank patients by risk so that limited intervention resources can be directed toward the most vulnerable cases.

Because of this, the project emphasizes:

- **Recall**, to reduce missed high-risk patients
- **F1**, in some cases for simplicity of the project
- **ROC-AUC**, to measure ranking quality
- **Lift and gains**, to evaluate prioritization effectiveness
- **Interpretability**, to understand what drives predictions
- **Fairness**, to ensure performance is reasonably consistent across patient groups

## Dataset and Data Understanding

This project uses the **Diabetes 130-US Hospitals for Years 1999–2008** dataset from the UCI Machine Learning Repository. The dataset contains hospital encounter records for patients diagnosed with diabetes and was designed for studying early readmission, especially readmission within 30 days of discharge. According to UCI, the dataset represents **10 years (1999–2008) of clinical care at 130 US hospitals and integrated delivery networks**, with **101,766 encounters** and **47 original features**. The feature set includes demographic information, admission/discharge information, diagnosis codes, laboratory results, medication-related variables, and prior utilization counts such as inpatient, outpatient, and emergency visits.

The original prediction target in this dataset is readmission status, including whether a patient was readmitted within 30 days, after 30 days, or not readmitted. For this project, the problem was reframed as a **binary risk prediction task** focused on identifying patients likely to be readmitted within 30 days, because that aligns more directly with a clinical intervention use case.

The dataset is well suited to this problem because it contains several classes of information that are clinically relevant to readmission risk:

- **Demographics:** age, gender, race  
- **Encounter context:** admission type, admission source, discharge disposition, time in hospital  
- **Clinical burden:** diagnosis categories, number of diagnoses, number of medications, diabetic medication indicators  
- **Lab and treatment signals:** HbA1c / A1C-related features, diabetes medication changes  
- **Prior utilization:** counts of inpatient, outpatient, and emergency encounters before hospitalization  

### Dataset source

UCI Machine Learning Repository: **Diabetes 130-US Hospitals for Years 1999–2008** 

## Dataset Cleaning and feature Engineering methods

### Target Definition

The original readmitted variable contained three categories: <30, >30, and NO. This was converted into a binary target:
1. 1 → readmitted within 30 days
2. 0 → not readmitted within 30 days

This transformation aligns the problem with a clinically actionable objective: identifying patients at risk of early readmission

### Leakage-Aware Data Splitting 

To prevent information leakage, the dataset was split into training and test sets using patient-level grouping (patient_nbr). 
This ensures that all encounters for a given patient appear in only one split.
This step is critical because multiple encounters from the same patient can otherwise lead to overly optimistic performance estimate

### Handling Missing and Sparse Variables

Several variables exhibited high missingness or limited informational value:
1. weight
2. payer_code

These were removed from the modeling dataset due to sparsity and low predictive utility. For other variables, missing or unknown values were retained and explicitly encoded (e.g., as "Unknown" categories) to preserve potentially meaningful signals.

Additionally, hospice/death discharges were removed because these patients cannot be readmitted and including them in the dataset would corrupt the target variable 

There were 3 rows that had gender = 'Invalid/Unknown'. These rows were removed as they cannot to categorized under patient demography

### Target Distribution

- **Not readmitted within 30 days:** 88.6%
- **Readmitted within 30 days:** 11.4%

Class imbalance addressed through cost-sensitive learning
(scale_pos_weight = 7.74) — equivalent to fraud modeling
practice where positive class is substantially rarer than
negative class.


### Categorical Grouping and Dimensionality Reduction

Many raw categorical variables had high cardinality or inconsistent coding. These were consolidated into clinically meaningful groups to improve model stability and interpretability.

Key transformations included:
1. Admission context: grouped admission_type_id and admission_source_id
2. Discharge context: grouped discharge_disposition_id into interpretable outcome categories
3. Medical specialty: reduced to medical_specialty_group, including an explicit "Unknown" category
4. Diagnosis variables: diag_1, diag_2, diag_3 mapped into broader diagnostic categories (e.g., respiratory, digestive, injury, neoplasms)

These transformations reduced sparsity while preserving clinically relevant structure.

# For detailed information on feature engineering please visit - [Other Notebook](.)

### Encoding and Final Dataset Construction
After grouping and cleaning, categorical variables were converted into model-ready form using one-hot encoding. This resulted in an expanded feature space where each category is represented as a binary indicator.
The final dataset includes:
A binary target variable for 30-day readmission
Patient-level train/test separation
Clinically grouped categorical variables
Utilization, diagnosis, treatment, and encounter intensity features
Removal of sparse and low-value predictors